# Teks — Ensemble OOF (gabung model dari notebook mana pun, dinilai dengan metrik lomba)
**Kapan dipakai:** sesudah ≥ 2 model teks menulis OOF ke `OOF_FOLDER` (`baseline_oof`,
`finetune_oof`, `teks_tabular`, ...). Notebook ini **tidak melatih model**: ia memuat
`oof_<MODEL>.npy` + `test_<MODEL>.npy`, memilih cara menggabung yang skor OOF-nya terbaik di
metrik lomba (bukan logloss saja), lalu menulis `submission.csv`.

**Alur:** Konfigurasi → metrik → baca → muat OOF (disaring per dataset) → skor + korelasi →
cara gabung (tunggal / rata-rata / hill climbing; alt rank / Nelder-Mead / stacking) →
**skor jujur** → pengali kelas → simpan + ablation → submission → validator.

**Skor jujur = bebas bocor bobot.** Bobot gabungan dicari di OOF yang sama dengan yang dinilai,
jadi skor "dalam sampel" optimistis. Skor jujur: bobot dicari di 4/5 baris OOF, dinilai di 1/5
sisanya (5 putaran). Pilihan akhir memakai skor jujur dan memilih cara PALING SEDERHANA yang
skornya tidak kalah dari yang terbaik lebih dari `MIN_GAIN`; kalau tidak ada gabungan yang
menang, notebook memakai model tunggal terbaik dan mengatakannya.

**Saat lomba yang diubah:** sel Konfigurasi (`# GANTI saat lomba`) dan sel Metrik. Hapus sel
bertanda `HAPUS saat lomba`.

**Data demo:** SmSA (`sentiment`) dengan OOF dari `baseline_oof` (dan `finetune_oof` bila ada).
Bentuk lain: `classification` (emot) cukup mengubah Konfigurasi.

In [ ]:
import glob
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar
from scipy.special import softmax
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.metrics import cohen_kappa_score, f1_score, log_loss, roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

## Konfigurasi
`OOF_FOLDER` sama dengan notebook lain. Folder itu bisa berisi OOF dari dataset lain: sel muat
hanya mengambil yang bentuknya cocok dengan data ini, lalu membuang yang skornya tidak lebih
baik dari tebakan konstan. Ragu? Isi `MODEL_NAMES` dengan daftar nama yang pasti sedataset.

In [ ]:
# ============ NILAI SOAL: isi SAMA di semua notebook 1_..7_ (tidak ada konfig.py) ============
# Sesudah mengubah: Restart Kernel + Run All. Nilai awal = data demo SmSA (sentimen, 3 kelas).
# ---- File soal (garis miring /, bukan \) ----
SAMPLE_PANITIA = None                         # GANTI: "C:/GEMASTIK/soal/sample_submission.csv"
#                                               None = sample demo dibuat sendiri (format BELUM dicek)
# ---- Kolom ----
ID_COL = "id"                                 # GANTI: kolom id
TEXT_COLS = ["text"]                          # GANTI: kolom teks. 4_finetune hanya memakai yang pertama
TARGET = "label"                              # GANTI: kolom kelas
SUBMIT_COL = TARGET                           # GANTI bila kolom prediksi di sample bernama lain
LABEL_COLS = None                             # MULTILABEL: daftar kolom label 0/1, mis. ["a", "b"].
#                                               None = single label. Bila diisi: pakai 3m, 4m, 5m
# ---- Hasil ----
HASIL = "../../outputs/final"                 # GANTI NAMA bila ganti dataset (mis. "../../outputs/lomba"),
#                                               supaya OOF demo tidak tercampur dengan OOF soal

# Coba jalur multilabel di data demo CASA (ulasan mobil, 6 label): buka # di empat baris ini
# TRAIN_MENTAH = "demo/casa_train.csv"
# TEST_MENTAH = "demo/casa_test.csv"
# LABEL_COLS = ["fuel", "machine", "others", "part", "price", "service"]
# HASIL = "../../outputs/final_multilabel"
# Multilabel: EDA, praproses, dan "topik x kelas" butuh SATU kolom kelas. Bila TARGET bukan salah satu
# LABEL_COLS, dipakai label pertama (ganti TARGET ke label yang paling penting).
if LABEL_COLS is not None and TARGET not in LABEL_COLS:
    TARGET = LABEL_COLS[0]
    SUBMIT_COL = TARGET
# ====================================================================================================

DATA_FOLDER = "../../data_latihan/text/sentiment"          # GANTI saat lomba
SUBMISSION_NAME = "submission.csv"                             # GANTI saat lomba bila diminta lain

TASK = "classification"          # GANTI: "regression" bila target angka (OOF 1 kolom)
MODEL_NAMES = None               # None = semua oof_*.npy yang cocok, atau ["tfidf_svc", "indobert"]

N_FOLDS = 5                      # putaran skor jujur (samakan dengan notebook penghasil OOF)
SEED = 42
MIN_GAIN = 0.002                 # gabungan/pengali dipakai hanya bila naik sekurangnya ini
WRITE_PSEUDO = False             # True = tulis CSV pseudo-label (test yakin) untuk finetune_oof
CONFIDENCE = 0.9                 # ambang yakin pseudo-label

# ---- jalur final-pipeline: dihitung dari HASIL di atas, tidak perlu diubah ----
TRAIN_CSV = f"{HASIL}/praproses/train_bersih.csv"
TEST_CSV = f"{HASIL}/praproses/test_bersih.csv"
OUTPUT_FOLDER = f"{HASIL}/ensemble"
OOF_FOLDER = f"{HASIL}/oof"
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"
if SAMPLE_PANITIA is not None:
    SAMPLE_CSV = SAMPLE_PANITIA

## Metrik
Aturan: `score(truth, pred)` **semakin besar semakin baik** (logloss dan RMSE dibalik tandanya).
`pred` = probabilitas (n, kelas) untuk klasifikasi, atau nilai (n,) untuk regresi. Ganti baris
`return` bila metrik panitia lain, dan `METRIC_NAME` ikut.

In [ ]:
METRIC_NAME = "f1_macro"         # GANTI saat lomba, ikut baris return di bawah


def score(truth, pred):
    return f1_score(truth, pred.argmax(axis=1), average="macro")
    # return accuracy_score(truth, pred.argmax(axis=1))                          # akurasi
    # return -log_loss(truth, pred, labels=range(pred.shape[1]))                 # logloss (dibalik)
    # return cohen_kappa_score(truth, pred.argmax(axis=1), weights="quadratic")  # QWK kelas ordinal
    # return roc_auc_score(truth, pred[:, 1])                                    # AUC biner
    # return roc_auc_score(truth, pred, multi_class="ovr")                       # AUC multikelas
    # return -np.sqrt(np.mean((truth - pred) ** 2))                              # RMSE (regresi)
    # return cohen_kappa_score(truth.round(), pred.round(), weights="quadratic") # QWK regresi

## Baca data
`y` = indeks kelas 0..K-1 (urut `np.unique` label train, sama dengan notebook penghasil OOF);
regresi: nilai asli.

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
if TASK == "classification":
    classes = np.unique(train[TARGET])
    y = np.searchsorted(classes, train[TARGET].to_numpy())
    oof_shape = (len(train), len(classes))
    test_shape = (len(test), len(classes))
else:
    classes = None
    y = train[TARGET].to_numpy(dtype=float)
    oof_shape = (len(train),)
    test_shape = (len(test),)
print("train:", train.shape, "| test:", test.shape, "| OOF diharapkan:", oof_shape)
print("kelas:", classes)

## Buat sample_submission demo (HAPUS saat lomba)
Data latihan tidak punya `sample_submission`. Saat lomba file panitia dipakai (`SAMPLE_CSV`);
sel ini tidak menimpa file di luar `OUTPUT_FOLDER`.

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUBMIT_COL] = train[TARGET].iloc[0]
if TASK == "regression":
    demo_sample[SUBMIT_COL] = 0.0
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):
    demo_sample.to_csv(SAMPLE_CSV, index=False)
    print("sample demo ditulis ke", SAMPLE_CSV)
else:
    print("SAMPLE_CSV bukan file demo: dibiarkan apa adanya")

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
print("sample_submission:", sample.shape, sample.columns.tolist())
print("id test = id sample:", set(test[ID_COL]) == set(sample[ID_COL]))
sample.head(3)

## Muat OOF dari `OOF_FOLDER`
Satu model = pasangan `oof_<nama>.npy` + `test_<nama>.npy`. Syarat masuk: pasangan lengkap,
bentuk sama dengan data ini (jumlah baris train/test dan jumlah kelas), semua angka terbatas,
tidak ada baris OOF nol (OOF split waktu belum lengkap). Baris yang jumlahnya bukan 1 dinormalkan
(mis. NB-SVM per kelas). Yang ditolak dicetak beserta alasannya.

In [ ]:
def normalize_rows(table):
    sums = table.sum(axis=1, keepdims=True)
    return table / np.clip(sums, 1e-12, None)


def read_problem(name):
    test_path = f"{OOF_FOLDER}/test_{name}.npy"
    if not os.path.exists(test_path):
        return "tidak ada test_*.npy", None, None
    oof = np.load(f"{OOF_FOLDER}/oof_{name}.npy").astype("float64")
    test_pred = np.load(test_path).astype("float64")
    if oof.shape != oof_shape or test_pred.shape != test_shape:
        return f"bentuk {oof.shape}/{test_pred.shape}, perlu {oof_shape}/{test_shape}", None, None
    if not (np.isfinite(oof).all() and np.isfinite(test_pred).all()):
        return "ada NaN/inf", None, None
    if TASK == "classification" and (oof.sum(axis=1) <= 0).any():
        return "OOF belum lengkap (ada baris nol)", None, None
    if TASK == "classification":
        return "", normalize_rows(oof), normalize_rows(test_pred)
    return "", oof, test_pred


print("pembaca OOF siap")

In [ ]:
oofs, tests, excluded = {}, {}, {}
for path in sorted(glob.glob(f"{OOF_FOLDER}/oof_*.npy")):
    name = os.path.basename(path)[4:-4]
    if MODEL_NAMES is not None and name not in MODEL_NAMES:
        continue
    reason, oof, test_pred = read_problem(name)
    if reason:
        excluded[name] = reason
    else:
        oofs[name], tests[name] = oof, test_pred
print("bentuk cocok:", list(oofs))
for name, reason in excluded.items():
    print(f"  ditolak {name}: {reason}")
if len(oofs) == 0:
    print("TIDAK ADA OOF YANG COCOK: jalankan baseline_oof dulu / cek TRAIN_CSV dan TASK")

## Skor tiap model dan saringan terakhir
Dua pembanding: tebakan konstan (kelas terbanyak / rata-rata) dan skor OOF yang BARISNYA DIACAK
(model sungguhan jauh di atasnya; OOF dataset lain yang kebetulan sebentuk sama dengan acak).
Model yang tidak melewati keduanya (dengan margin 10% + 0,005) dikeluarkan, kecuali
`MODEL_NAMES` diisi, karena itu pilihan sendiri.

In [ ]:
def constant_prediction():
    if TASK == "classification":
        prior = np.bincount(y, minlength=len(classes)) / len(y)
        return np.tile(prior, (len(y), 1))
    return np.full(len(y), y.mean())


baseline_score = score(y, constant_prediction())
shuffled = np.random.RandomState(SEED).permutation(len(y))
rows = []
for name in list(oofs):
    value = score(y, oofs[name])
    null_value = score(y, oofs[name][shuffled])
    bar = max(baseline_score, null_value) + 0.1 * abs(null_value) + 0.005
    if MODEL_NAMES is None and value <= bar:
        excluded[name] = f"skor {value:.4f} <= acak {null_value:.4f} (OOF dataset lain?)"
        print(f"  dikeluarkan {name}: {excluded[name]}")
        del oofs[name]
        del tests[name]
    else:
        rows.append({"model": name, f"{METRIC_NAME}_oof": round(value, 4),
                     "baris_diacak": round(null_value, 4)})
print(f"{METRIC_NAME} tebakan konstan: {baseline_score:.4f}")
single_table = pd.DataFrame(rows).sort_values(f"{METRIC_NAME}_oof", ascending=False)
print(single_table.to_string(index=False))

## Korelasi antar model
Klasifikasi: korelasi proba untuk kelas yang benar; regresi: korelasi galat. Makin rendah makin
berguna digabung; model kuat dengan korelasi ≈ 0,99 hampir tidak menambah apa-apa.

In [ ]:
names = list(oofs)
errors = {}
for name in names:
    if TASK == "classification":
        errors[name] = oofs[name][np.arange(len(y)), y]
    else:
        errors[name] = oofs[name] - y
correlation = pd.DataFrame(errors).corr()
print(correlation.round(3))
stack_list, test_list = [], []
for name in names:
    stack_list.append(oofs[name])
    test_list.append(tests[name])
stack_oof = np.stack(stack_list)      # (model, baris train[, kelas])
stack_test = np.stack(test_list)      # (model, baris test[, kelas])
print("tumpukan OOF:", stack_oof.shape, "| test:", stack_test.shape)

## Cara menggabung
Tiap cara = fungsi `(fit_stack, truth, apply_stacks)` → (daftar hasil, keterangan): bobot
dicari di `fit_stack` lalu dipakai ke tiap tumpukan di `apply_stacks`. Dengan antarmuka yang
sama, skor jujur (bobot dari 4/5 baris, nilai di 1/5) berlaku untuk semua cara.

- **tunggal**: model terbaik saja (patokan; gabungan harus mengalahkannya).
- **rata**: rata-rata proba semua model.
- **hill**: ensemble selection Caruana dengan pengembalian: mulai dari model terbaik, tiap langkah
  tambah model yang paling menaikkan metrik, berhenti bila tidak naik.

In [ ]:
def blend_with(weights, stack):
    return np.tensordot(weights, stack, axes=1)


def apply_weights(weights, apply_stacks):
    outputs = []
    for stack in apply_stacks:
        outputs.append(blend_with(weights, stack))
    return outputs


def describe(weights):
    parts = []
    for name, weight in zip(names, weights):
        if weight > 0:
            parts.append(f"{name} {weight:.2f}")
    return ", ".join(parts)


print("alat gabung siap")

In [ ]:
def single_method(fit_stack, truth, apply_stacks):
    values = []
    for model_pred in fit_stack:
        values.append(score(truth, model_pred))
    weights = np.zeros(len(fit_stack))
    weights[int(np.argmax(values))] = 1.0
    return apply_weights(weights, apply_stacks), describe(weights)


def mean_method(fit_stack, truth, apply_stacks):
    weights = np.ones(len(fit_stack)) / len(fit_stack)
    return apply_weights(weights, apply_stacks), "bobot sama"


print("tunggal dan rata siap")

In [ ]:
def hill_weights(fit_stack, truth, max_steps=30):
    counts = np.zeros(len(fit_stack))
    values = []
    for model_pred in fit_stack:
        values.append(score(truth, model_pred))
    counts[int(np.argmax(values))] = 1
    best = max(values)
    for _ in range(max_steps):
        trials = []
        for number in range(len(fit_stack)):
            trial = counts.copy()
            trial[number] += 1
            trials.append(score(truth, blend_with(trial / trial.sum(), fit_stack)))
        pick = int(np.argmax(trials))
        if trials[pick] <= best:
            break
        counts[pick] += 1
        best = trials[pick]
    return counts / counts.sum()


def hill_method(fit_stack, truth, apply_stacks):
    weights = hill_weights(fit_stack, truth)
    return apply_weights(weights, apply_stacks), describe(weights)


print("hill climbing siap")

**Alternatif** (fungsinya siap; aktifkan di sel `METHODS` dengan menghapus `#`):

- **rank**: rata-rata PERINGKAT per model dan kelas. Untuk AUC/PR-AUC atau model beda skala.
  Untuk F1/akurasi multikelas sering LEBIH BURUK (peringkat per kelas membuang sebaran kelas);
  hasilnya bukan proba terkalibrasi, jangan dinilai dengan logloss. Peringkat test dihitung di
  test sendiri.
- **nelder_mead**: optimasi bobot (softmax) langsung ke metrik, `scipy.optimize`. Sedikit model
  (≤ 6) dan metrik halus (logloss, RMSE, AUC); untuk F1 (bertingkat) kurang stabil.
- **stacking**: LogReg (regresi: Ridge) di atas log-proba semua model. Model beragam dan OOF
  ≥ beberapa ribu baris; level 2 harus sederhana.

In [ ]:
def nelder_mead_weights(fit_stack, truth):
    def loss(params):
        return -score(truth, blend_with(softmax(params), fit_stack))

    start = np.zeros(len(fit_stack))
    found = minimize(loss, start, method="Nelder-Mead", options={"maxiter": 300})
    return softmax(found.x)


def nelder_mead_method(fit_stack, truth, apply_stacks):
    weights = nelder_mead_weights(fit_stack, truth)
    return apply_weights(weights, apply_stacks), describe(weights)


def rank_method(fit_stack, truth, apply_stacks):
    outputs = []
    for stack in apply_stacks:
        ranks = rankdata(stack, axis=1) / stack.shape[1]
        merged = ranks.mean(axis=0)
        if TASK == "classification":
            merged = normalize_rows(merged)
        outputs.append(merged)
    return outputs, "rata-rata peringkat"


print("rank dan Nelder-Mead siap (alt)")

In [ ]:
def stack_features(stack):
    if TASK != "classification":
        return stack.T
    parts = []
    for model_pred in stack:
        parts.append(np.log(np.clip(model_pred, 1e-6, 1)))
    return np.hstack(parts)


def stack_method(fit_stack, truth, apply_stacks):
    level_two = Ridge(alpha=1.0)
    if TASK == "classification":
        level_two = LogisticRegression(C=0.1, max_iter=3000)
    level_two.fit(stack_features(fit_stack), truth)
    outputs = []
    for stack in apply_stacks:
        if TASK == "classification":
            outputs.append(level_two.predict_proba(stack_features(stack)))
        else:
            outputs.append(level_two.predict(stack_features(stack)))
    return outputs, "LogReg/Ridge di atas log-proba"


print("stacking siap (alt)")

## Pilih cara yang dibandingkan
Urutan = dari yang paling sederhana; pada skor jujur yang sama-sama dekat, yang lebih awal menang.

In [ ]:
METHODS = {"tunggal": single_method, "rata": mean_method, "hill": hill_method}
# METHODS["rank"] = rank_method                  # AUC / model beda skala (F1 multikelas: hati-hati)
# METHODS["nelder_mead"] = nelder_mead_method    # model sedikit, metrik halus (logloss, RMSE)
# METHODS["stacking"] = stack_method             # model beragam, OOF besar; level 2 sederhana
print("dibandingkan:", list(METHODS))

## Skor jujur
Putaran luar `outer_folds` memakai seed `SEED + 1` (bukan seed pembuat OOF). Tiap putaran:
bobot dicari di baris latih putaran itu, dipakai ke baris validasinya. Skor "dalam sampel" =
bobot dari semua baris dipakai ke semua baris (optimistis, hanya pembanding).

In [ ]:
def outer_folds():
    if TASK == "classification":
        return list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + 1).split(y, y))
    return list(KFold(N_FOLDS, shuffle=True, random_state=SEED + 1).split(y))


def honest_prediction(method):
    honest = np.zeros_like(stack_oof[0])
    for fit_index, val_index in folds_outer:
        outputs, _ = method(stack_oof[:, fit_index], y[fit_index], [stack_oof[:, val_index]])
        honest[val_index] = outputs[0]
    return honest


folds_outer = outer_folds()
honest_preds, full_preds, honest_scores, infos, rows = {}, {}, {}, {}, []
ensemble_start = time.time()
for method_name, method in METHODS.items():
    start = time.time()
    honest_preds[method_name] = honest_prediction(method)
    full_preds[method_name], infos[method_name] = method(stack_oof, y, [stack_oof, stack_test])
    honest_scores[method_name] = score(y, honest_preds[method_name])
    rows.append({"cara": method_name, "skor_dalam_sampel": score(y, full_preds[method_name][0]),
                 "skor_jujur": honest_scores[method_name],
                 "detik": round(time.time() - start, 1), "keterangan": infos[method_name]})
board = pd.DataFrame(rows)
board = board.sort_values("skor_jujur", ascending=False)
print(board.round(4).to_string(index=False))

## Putusan
Pilih cara PALING SEDERHANA (urutan `METHODS`) yang skor jujurnya tidak kalah dari yang terbaik
lebih dari `MIN_GAIN`. `tunggal` paling awal: kalau gabungan tidak menang, model tunggal dipakai.

In [ ]:
best_value = max(honest_scores.values())
chosen = None
for method_name in METHODS:
    if chosen is None and honest_scores[method_name] >= best_value - MIN_GAIN:
        chosen = method_name
final_honest = honest_preds[chosen]
final_oof, final_test = full_preds[chosen]
print(f"dipilih: {chosen} ({infos[chosen]}) | skor jujur {honest_scores[chosen]:.4f} "
      f"| terbaik {best_value:.4f}")
if chosen == "tunggal":
    print("TIDAK ADA gabungan yang mengalahkan model tunggal terbaik: dipakai model tunggal itu")

## Kalibrasi sesudah gabung (alt: temperature scaling)
Hanya untuk metrik logloss/proba (argmax tidak berubah): cari satu angka `T` di OOF yang
meminimalkan logloss, `proba^(1/T)` dinormalkan. Hapus `#` di empat baris terakhir sel ini.

In [ ]:
def sharpen(pred, temperature):
    return softmax(np.log(np.clip(pred, 1e-9, 1)) / temperature, axis=1)


def fit_temperature(pred, truth):
    def loss(temperature):
        return log_loss(truth, sharpen(pred, temperature), labels=range(pred.shape[1]))

    return minimize_scalar(loss, bounds=(0.2, 5), method="bounded").x


# temperature = fit_temperature(final_honest, y)    # model terlalu yakin (T > 1) atau ragu (T < 1)
# final_honest = sharpen(final_honest, temperature)
# final_test = sharpen(final_test, temperature)
# print("temperature:", round(temperature, 3))
print("temperature scaling: nonaktif (alt)")

## Pengali kelas (F1 makro)
Pengali per kelas dicari dengan coordinate ascent pada hasil gabungan. Dipakai hanya bila naik
≥ `MIN_GAIN` pada skor JUJUR (pengali dicari di 4/5 baris, dinilai di 1/5). Hanya klasifikasi.

In [ ]:
def reweigh(pred, weights):
    return normalize_rows(pred * weights)


def optimize_class_weights(pred, truth, rounds=2):
    weights = np.ones(pred.shape[1])
    best = score(truth, pred)
    for _ in range(rounds):
        for class_number in range(len(weights)):
            for candidate in [0.5, 0.75, 1, 1.5, 2, 3, 5, 8]:
                trial = weights.copy()
                trial[class_number] = candidate
                trial_score = score(truth, reweigh(pred, trial))
                if trial_score > best:
                    best, weights = trial_score, trial
    return weights, best


print("fungsi pengali siap")

In [ ]:
class_weights = None
use_weights = False
result_honest, result_test = final_honest, final_test
if TASK == "classification":
    weighted_honest = np.zeros_like(final_honest)
    for fit_index, val_index in folds_outer:
        found, _ = optimize_class_weights(final_honest[fit_index], y[fit_index])
        weighted_honest[val_index] = reweigh(final_honest[val_index], found)
    honest_gain = score(y, weighted_honest) - score(y, final_honest)
    class_weights, _ = optimize_class_weights(final_honest, y)
    use_weights = honest_gain >= MIN_GAIN
    print("pengali:", class_weights.round(2), f"| naik (jujur) {honest_gain:.4f}")
    print("pengali DIPAKAI:", use_weights, f"(syarat naik >= {MIN_GAIN})")
if use_weights:
    result_honest = weighted_honest
    result_test = reweigh(final_test, class_weights)
final_score = score(y, result_honest)
print(f"{METRIC_NAME} jujur hasil akhir: {final_score:.4f}")

## Evaluasi hasil akhir
Prediksi OOF jujur (bobot tidak pernah melihat baris yang dinilai).

In [ ]:
if TASK == "classification":
    result_pred = result_honest.argmax(axis=1)
    print(classification_report(y, result_pred, target_names=classes.astype(str)))
    ConfusionMatrixDisplay.from_predictions(y, result_pred, display_labels=classes,
                                            normalize="true", xticks_rotation=45)
    plt.show()
else:
    print("RMSE jujur:", np.sqrt(np.mean((y - result_honest) ** 2)).round(4))

## Simpan dan catat percobaan
Hasil gabungan disimpan di `OUTPUT_FOLDER` (BUKAN `OOF_FOLDER`, supaya tidak ikut digabung lagi).
`percobaan.csv` bertambah satu baris tiap run.

In [ ]:
np.save(f"{OUTPUT_FOLDER}/blend_oof_jujur.npy", result_honest.astype("float32"))
np.save(f"{OUTPUT_FOLDER}/blend_test.npy", result_test.astype("float32"))
log_path = f"{OOF_FOLDER}/percobaan.csv"
note = (f"{len(names)} model {names} | cara {chosen}: {infos[chosen]} | pengali {use_weights} | "
        f"dalam sampel {score(y, final_oof):.4f}")
row = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "notebook": "ensemble_oof",
       "model": f"ensemble_{chosen}", "metrik": METRIC_NAME, "skor_cv": round(final_score, 4),
       "detik": round(time.time() - ensemble_start, 1), "catatan": note}
pd.DataFrame([row]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
print("disimpan:", f"{OUTPUT_FOLDER}/blend_oof_jujur.npy", "+ blend_test.npy")

## Tabel ablation (bahan slide)
Tiap langkah dan skornya di OOF yang sama; selisih dibanding model tunggal terbaik. Disimpan ke
`ablation.csv`. Di bawahnya: skor OOF yang dihitung ulang dari file dibanding baris terakhir
`percobaan.csv` (beda besar = baris itu dari dataset/metrik lain).

In [ ]:
ablation = []
for name in names:
    ablation.append({"langkah": f"model {name}", "skor": score(y, oofs[name])})
best_single = max(single_table[f"{METRIC_NAME}_oof"])
for method_name in METHODS:
    ablation.append({"langkah": f"cara {method_name} (jujur)", "skor": honest_scores[method_name]})
ablation.append({"langkah": f"dipilih {chosen} + pengali kelas (jujur)", "skor": final_score})
ablation = pd.DataFrame(ablation)
ablation["selisih_vs_tunggal_terbaik"] = ablation["skor"] - best_single
ablation.to_csv(f"{OUTPUT_FOLDER}/ablation.csv", index=False)
print(ablation.round(4).to_string(index=False))

In [ ]:
history = pd.read_csv(log_path)
for name in names:
    logged = history[history["model"] == name]["skor_cv"]
    if len(logged) == 0:
        print(f"  {name}: tidak ada baris di percobaan.csv")
    elif abs(logged.iloc[-1] - score(y, oofs[name])) > 0.002:
        print(f"  {name}: percobaan {logged.iloc[-1]:.4f} vs dihitung {score(y, oofs[name]):.4f}")
print("selesai cek percobaan.csv (hanya baris yang beda dicetak)")

## CSV pseudo-label untuk `finetune_oof` (opsional)
Isi `WRITE_PSEUDO = True`: teks test dengan keyakinan ≥ `CONFIDENCE` ditulis ke
`pseudo_label.csv` (kolom id, teks, target seperti train) supaya `finetune_oof` menggabungnya
dengan `pd.concat`. Hanya klasifikasi. Dinilai di CV `finetune_oof`: sering tidak menolong
bila test kecil.

In [ ]:
if WRITE_PSEUDO and TASK == "classification":
    confident = result_test.max(axis=1) >= CONFIDENCE
    pseudo = test.loc[confident, [ID_COL] + TEXT_COLS].copy()
    pseudo[TARGET] = classes[result_test[confident].argmax(axis=1)]
    pseudo.to_csv(f"{OUTPUT_FOLDER}/pseudo_label.csv", index=False)
    print(f"pseudo-label: {confident.sum()} dari {len(test)} test yakin >= {CONFIDENCE}")
    print(pseudo[TARGET].value_counts())
else:
    print("pseudo-label: dilewati (WRITE_PSEUDO = False atau regresi)")

## Submission
Kolom id diambil dari `sample_submission`; prediksi dipetakan lewat id.

In [ ]:
if TASK == "classification":
    values = classes[result_test.argmax(axis=1)]
else:
    values = result_test
pred_by_id = pd.Series(values, index=test[ID_COL])

submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUBMIT_COL] = sample[ID_COL].map(pred_by_id)
# soal minta probabilitas per kelas: ganti baris di atas dengan loop
# proba_by_id = pd.DataFrame(result_test, columns=classes.astype(str), index=test[ID_COL])
# proba_by_id = proba_by_id.reindex(sample[ID_COL]).reset_index(drop=True)
# submission = pd.concat([submission[[ID_COL]], proba_by_id], axis=1)
submission.to_csv(f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}", index=False)
print(submission[SUBMIT_COL].describe())
submission.head()

## Cek submission terhadap sample_submission
Salinan validator T3 (`arsenal/umum/simpan_muat` #15 dan #16): hanya membaca dan mencetak. Hasil
akhir harus `MASALAH: 0`. Dicek: kolom dan urutannya, jumlah baris, id hilang / asing / kembar /
urutan beda, nilai kosong, tipe kolom ("3" vs "3.0"), label di luar `ALLOWED`.

In [ ]:
SUBMISSION_CSV = f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}"
ID_COLUMN = ID_COL

sample = pd.read_csv(SAMPLE_CSV)
written = pd.read_csv(SUBMISSION_CSV)
problems = []


def report(failed, text):
    if failed:
        problems.append(text)


wanted = sample[ID_COLUMN]
ids = written.get(ID_COLUMN, pd.Series(dtype=wanted.dtype))
lost = set(wanted) - set(ids)
extra = set(ids) - set(wanted)
kept = ids[ids.isin(wanted)].drop_duplicates().tolist()
expected = wanted[wanted.isin(ids)].tolist()
report(written.columns.tolist() != sample.columns.tolist(), "kolom / urutan kolom beda")
report(len(written) != len(sample), f"jumlah baris {len(written)}, sample {len(sample)}")
report(lost, f"id hilang {len(lost)}, mis. {sorted(lost)[:3]}")
report(extra, f"id asing {len(extra)}, mis. {sorted(extra)[:3]}")
report(ids.duplicated().any(), f"id kembar: {ids.duplicated().sum()} baris")
report(kept != expected, "urutan id beda dari sample")

In [ ]:
LABEL_COLUMNS = [SUBMIT_COL]     # kolom isi kelas (soal proba per kelas: isi [] untuk lewati cek)
ALLOWED = []                     # label sah = label train, [] = lewati cek (regresi)
if TASK == "classification":
    ALLOWED = list(classes)

for column in sample.columns:
    if column in written.columns:
        empty = written[column].isna().sum()
        report(empty, f"{column}: {empty} nilai kosong")
        report(written[column].dtype != sample[column].dtype,
               f"{column}: tipe {written[column].dtype}, sample {sample[column].dtype}")
for column in LABEL_COLUMNS:
    if ALLOWED and column in written.columns:
        foreign = set(written[column].dropna()) - set(ALLOWED)
        report(foreign, f"{column}: label di luar ALLOWED {sorted(foreign)[:5]}")
        print(written[column].value_counts())   # semua satu kelas -> curigai model
for problem in problems:
    print("-", problem)
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):
    print("PERINGATAN: SAMPLE_CSV masih file demo, format panitia belum dicek")
print("MASALAH:", len(problems))